# 01 — EDA: Thai Government Lottery Data

**Project Fortuna — Phase 1**

Exploratory analysis of draws.jsonl. Tests for uniformity, autocorrelation, and other patterns.

**Honest framing:** If signal exists, it should appear here. We expect most tests to *fail to reject H0* (i.e., the lottery looks random). Any p < 0.05 finding is anomalous and warrants Phase 2 investigation — but requires BH-FDR correction (SPEC §7.2) before claiming significance.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root on path
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

from fortuna.store import DrawStore
from fortuna.config import DRAWS_JSONL

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

In [ ]:
# Load draws
store = DrawStore(DRAWS_JSONL)
draws = store.all_draws()
draws.sort(key=lambda d: d.draw_id)

print(f"Total draws: {len(draws)}")
print(f"Date range: {draws[0].draw_date} to {draws[-1].draw_date}")

first_prizes = [d.first_prize for d in draws]
two_digit_backs = [d.two_digit_back for d in draws]
three_digit_backs_flat = [n for d in draws for n in d.three_digit_back]

## 1. Digit Position Frequency — First Prize

In [ ]:
# Count digit occurrences per position
digit_freqs = [Counter() for _ in range(6)]
for fp in first_prizes:
    for pos, ch in enumerate(fp):
        digit_freqs[pos][ch] += 1

# Chi-square uniformity test per position
chi_results = []
for pos in range(6):
    observed = [digit_freqs[pos].get(str(d), 0) for d in range(10)]
    n = sum(observed)
    expected = [n / 10.0] * 10
    chi2, p_val = stats.chisquare(observed, f_exp=expected)
    chi_results.append({'position': pos+1, 'n': n, 'chi2': chi2, 'p_value': p_val, 'reject_h0': p_val < 0.05})

chi_df = pd.DataFrame(chi_results)
print("Chi-square uniformity test — First Prize digit positions")
print(chi_df.to_string(index=False))
print()
print("H0: each digit 0-9 equally likely (uniform)")
print(f"Positions rejecting H0 at alpha=0.05: {[r['position'] for r in chi_results if r['reject_h0']]}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('First Prize — Digit Frequency by Position', fontsize=14)

for pos in range(6):
    ax = axes[pos // 3][pos % 3]
    digits = [str(d) for d in range(10)]
    counts = [digit_freqs[pos].get(d, 0) for d in digits]
    total = sum(counts)
    ax.bar(digits, counts, color='steelblue', alpha=0.7)
    ax.axhline(total / 10, color='red', linestyle='--', label='Expected')
    r = chi_results[pos]
    ax.set_title(f'Position {pos+1}  χ²={r["chi2"]:.2f}, p={r["p_value"]:.3f}')
    ax.set_xlabel('Digit')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/reports/figures/digit_position_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Two-Digit Suffix Distribution

In [ ]:
two_digit_counter = Counter(two_digit_backs)
values = [two_digit_counter.get(f'{i:02d}', 0) for i in range(100)]
expected_count = len(draws) / 100

chi2_2d, p_2d = stats.chisquare(values, f_exp=[expected_count] * 100)
print(f"Two-digit suffix χ² = {chi2_2d:.2f}, p = {p_2d:.4f}")
print(f"Reject H0: {p_2d < 0.05}")

plt.figure(figsize=(16, 4))
plt.bar(range(100), values, color='coral', alpha=0.7)
plt.axhline(expected_count, color='navy', linestyle='--', label=f'Expected ({expected_count:.1f})')
plt.title(f'Two-Digit Suffix Frequency  χ²={chi2_2d:.2f}, p={p_2d:.4f}')
plt.xlabel('Two-digit suffix (00-99)')
plt.ylabel('Count')
plt.xticks(range(0, 100, 10), [f'{i:02d}' for i in range(0, 100, 10)])
plt.legend()
plt.tight_layout()
plt.savefig('../data/reports/figures/two_digit_suffix_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Autocorrelation of Digit Sums

In [ ]:
digit_sums = np.array([sum(int(c) for c in fp) for fp in first_prizes], dtype=float)
centered = digit_sums - digit_sums.mean()
n = len(centered)
max_lag = min(30, n // 2)

acf_values = []
for lag in range(1, max_lag + 1):
    acf = np.correlate(centered[:n-lag], centered[lag:])[0] / np.sum(centered**2)
    acf_values.append(acf)

ci = 1.96 / np.sqrt(n)

plt.figure(figsize=(12, 5))
plt.bar(range(1, len(acf_values)+1), acf_values, color='teal', alpha=0.7)
plt.axhline(ci, color='red', linestyle='--', label=f'95% CI ±{ci:.3f}')
plt.axhline(-ci, color='red', linestyle='--')
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Autocorrelation of First Prize Digit Sum (Lag 1-30)')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.legend()
plt.tight_layout()
plt.savefig('../data/reports/figures/digit_sum_autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()

outside_ci = [(i+1, v) for i, v in enumerate(acf_values) if abs(v) > ci]
print(f"ACF values outside 95% CI: {outside_ci}")

## 4. Runs Test

In [ ]:
median_sum = np.median(digit_sums)
signs = [1 if s >= median_sum else 0 for s in digit_sums]
n1 = sum(signs)
n2 = len(signs) - n1
runs = 1
for i in range(1, len(signs)):
    if signs[i] != signs[i-1]:
        runs += 1

expected_runs = (2 * n1 * n2) / (n1 + n2) + 1
var_runs = (2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / ((n1 + n2)**2 * (n1 + n2 - 1))
z_runs = (runs - expected_runs) / (var_runs**0.5)
p_runs = 2 * (1 - stats.norm.cdf(abs(z_runs)))

print(f"Runs test on digit sums (above/below median={median_sum:.1f}):")
print(f"  n1 (above) = {n1}, n2 (below) = {n2}")
print(f"  Observed runs = {runs}")
print(f"  Expected runs = {expected_runs:.2f}")
print(f"  Z = {z_runs:.4f}, p = {p_runs:.4f}")
print(f"  Reject H0 (p<0.05): {p_runs < 0.05}")

## 5. Summary

In [ ]:
print('=' * 60)
print('PHASE 1 EDA SUMMARY')
print('=' * 60)
print(f'Total draws analyzed: {len(draws)}')
print(f'Date range: {draws[0].draw_date} → {draws[-1].draw_date}')
print()
print('Digit position chi-square uniformity:')
for r in chi_results:
    flag = ' <-- ANOMALY' if r['reject_h0'] else ''
    print(f'  Position {r["position"]}: χ²={r["chi2"]:.2f}, p={r["p_value"]:.4f}{flag}')
print()
print(f'Two-digit suffix: χ²={chi2_2d:.2f}, p={p_2d:.4f}{" <-- ANOMALY" if p_2d < 0.05 else ""}')
print(f'Runs test: Z={z_runs:.4f}, p={p_runs:.4f}{" <-- ANOMALY" if p_runs < 0.05 else ""}')
print()
print('Note: Multiple comparisons — BH-FDR correction required before any edge claim (SPEC §7.2)')